In [21]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Loaded successfully 🎉")


Loaded successfully 🎉


In [22]:
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [23]:
import numpy as np

In [24]:
poster_embeddings = np.load(r"D:\MINI PROJECT\NOTEBOOKS\poster_embeddings.npy")

norms = np.linalg.norm(poster_embeddings, axis=1)
print("Min norm:", norms.min())
print("Max norm:", norms.max())
print("Non-zero:", np.sum(norms > 0))


Min norm: 0.0
Max norm: 24.789915
Non-zero: 9056


In [25]:
# -------------------------------
# PATHS (EDIT ONLY IF NEEDED)
# -------------------------------
NCF_MODEL_PATH = r"D:\MINI PROJECT\NOTEBOOKS\ncfmodel.keras"
USER_ENCODER_PATH = r"D:\MINI PROJECT\NOTEBOOKS\user_encoder.pkl"
MOVIE_ENCODER_PATH = r"D:\MINI PROJECT\NOTEBOOKS\movie_encoder.pkl"
EMBEDDINGS_PATH = r"D:\MINI PROJECT\NOTEBOOKS\movie_embeddings.npy"

MOVIES_PATH = r"D:\MINI PROJECT\DATASET\movies_final.csv"
RATINGS_PATH = r"D:\MINI PROJECT\DATASET\MovieLensDataset20M\rating.csv"

# -------------------------------
# LOAD DATA
# -------------------------------
print("Loading data...")
movies = pd.read_csv(MOVIES_PATH)
ratings = pd.read_csv(RATINGS_PATH)

# -------------------------------
# LOAD MODEL & ENCODERS
# -------------------------------
print("Loading NCF model...")
ncfmodel = load_model(NCF_MODEL_PATH)

print("Loading encoders...")
user_encoder = joblib.load(USER_ENCODER_PATH)
movie_encoder = joblib.load(MOVIE_ENCODER_PATH)

# -------------------------------
# LOAD OR CREATE EMBEDDINGS
# -------------------------------
try:
    embeddings = np.load(EMBEDDINGS_PATH)
    print("Loaded existing embeddings:", embeddings.shape)

    # sanity check
    if embeddings.shape[0] != len(movies):
        raise ValueError("Embedding count mismatch")

except Exception as e:
    print("Rebuilding embeddings due to error:", e)

    text_model = SentenceTransformer("all-MiniLM-L6-v2")

    movies["content_text"] = (
        movies["title"].fillna("") + " " +
        movies["genres"].fillna("") + " " +
        movies["overview"].fillna("")
    )

    embeddings = text_model.encode(
        movies["content_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save(EMBEDDINGS_PATH, embeddings)
    print("Saved embeddings:", embeddings.shape)

Loading data...
Loading NCF model...
Loading encoders...
Loaded existing embeddings: (23138, 384)


In [26]:
text_embeddings=np.load("D:\MINI PROJECT\movie_embeddings.npy")
poster_embeddings=np.load(r"D:\MINI PROJECT\NOTEBOOKS\poster_embeddings.npy")

In [27]:
assert len(movies) == text_embeddings.shape[0]
assert len(movies) == poster_embeddings.shape[0]


In [28]:
#Poster embedding quality
norms = np.linalg.norm(poster_embeddings, axis=1)
print("Zero posters:", (norms == 0).sum())
print("Non-zero posters:", (norms > 0).sum())


Zero posters: 14082
Non-zero posters: 9056


In [29]:
idx = np.where(np.linalg.norm(poster_embeddings, axis=1) > 0)[0][0]
print("Sample norm:", np.linalg.norm(poster_embeddings[idx]))
print("Movie:", movies.iloc[idx]["title"])


Sample norm: 13.5267105
Movie: Toy Story


In [30]:
print("Type:", type(poster_embeddings))
print("Shape:", poster_embeddings.shape)
print("Dtype:", poster_embeddings.dtype)


Type: <class 'numpy.ndarray'>
Shape: (23138, 2048)
Dtype: float32
